# Mesurer le modèle de base sur le service client

Le fil rouge de la formation : le service client de Biclou & Cie, une chaîne (inventée) de six magasins de vélos. À chaque message d'un client, l'équipe trie la demande (catégorie, priorité) et rédige la réponse selon sa charte. Le but de la formation : qu'un petit modèle ouvert, Qwen3-1.7B, fasse les deux seul, en un objet JSON. Avant de l'affiner, on mesure ce qu'il vaut sans entraînement : c'est le point de référence que l'affinage devra battre.

1. La consigne seule (*zero-shot*)
2. Quelques exemples (*few-shot*)
3. La charte complète dans le prompt
4. Ce que coûte chaque prompt

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même, en plus petit et en une quinzaine de minutes : Qwen3-0.6B au lieu de Qwen3-1.7B, sur 4 tickets de test au lieu de 150. Sur 4 tickets, les mesures varient beaucoup : comparez les approches entre elles, plutôt qu'aux chiffres d'un voisin.

## Données et modèle

`test.jsonl` : 150 tickets validés par l'équipe, avec la réponse attendue. `charte.md` : la charte de réponse du service client. `tickets.csv` : l'export brut de l'outil de support, d'où viennent les exemples de l'exercice 2 (le TP suivant le nettoie).

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/test.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/tickets.csv

In [ ]:
import json
import re
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "Pas de GPU : Qwen3-0.6B, sur 4 tickets")

MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODELE)
tokenizer.padding_side = "left"
modele = AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)

test = pd.read_json("test.jsonl", lines=True)
test_part = test[: None if GPU else 4]
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()
tickets = pd.read_csv("tickets.csv")
print(len(test), "tickets de test ;", len(test_part), "mesurés ici")
test.head()

## La tâche et sa mesure

Le modèle reçoit une consigne courte, `CONSIGNES` (celle que recevra le modèle affiné), et le message du client. Il répond par un objet JSON à trois clés :

```json
{"categorie": "livraison", "priorite": "normale", "reponse": "Bonjour Madame Martin, ..."}
```

`evalue(sorties)` compare ses réponses à celles de `test_part` :

- `json_valide` : la part de réponses qui sont un objet JSON avec les trois clés ;
- `categorie`, `priorite` : la part de bonnes valeurs (exactitude) ;
- `charte` : la part de réponses qui suivent les règles vérifiables de la charte : la salutation, le vouvoiement, entre 40 et 120 mots, la signature ;
- `mots` : la longueur moyenne de la réponse, en mots.

Que la réponse dise **juste** (le bon délai, la bonne marche à suivre) ne se mesure pas ainsi : c'est le travail du LLM juge, dans l'atelier d'évaluation.

In [ ]:
CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
CONSIGNES = f"""Tu es l'assistant du service client de Biclou & Cie, magasins de vélos.
Pour le message du client, réponds uniquement par un objet JSON avec trois clés :
- categorie : une valeur parmi {", ".join(CATEGORIES)} ;
- priorite : basse, normale ou haute ;
- reponse : la réponse à envoyer au client."""
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def lit_json(texte: str) -> dict:
  """L'objet JSON d'une réponse du modèle, ou {} s'il n'est pas valide."""
  texte = texte.strip().removeprefix("```json").removesuffix("```").strip()
  try:
    objet = json.loads(texte)
  except json.JSONDecodeError:
    return {}
  return objet if isinstance(objet, dict) else {}


def respecte_charte(reponse: str) -> bool:
  reponse = reponse.strip()
  return (
    reponse.startswith("Bonjour")
    and reponse.endswith(SIGNATURE)
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


def evalue(sorties: list[str]) -> pd.Series:
  objets = [lit_json(s) for s in sorties]
  reponses = [str(o.get("reponse", "")) for o in objets]
  return (
    pd.Series(
      {
        "json_valide": sum(
          {"categorie", "priorite", "reponse"} <= o.keys() for o in objets
        ),
        "categorie": sum(
          o.get("categorie") == c for o, c in zip(objets, test_part["categorie"])
        ),
        "priorite": sum(
          o.get("priorite") == p for o, p in zip(objets, test_part["priorite"])
        ),
        "charte": sum(respecte_charte(r) for r in reponses),
        "mots": sum(len(r.split()) for r in reponses),
      }
    )
    .div(len(sorties))
    .round(2)
  )


print(CONSIGNES)

## 1. La consigne seule

Écrivez `genere_tout(conversations)`, qui génère par lots de 8 la réponse du modèle à chaque conversation (une liste de messages) et renvoie les textes. Puis mesurez le modèle sur `test_part`, avec pour chaque ticket une conversation de deux messages : `CONSIGNES` en message `system`, le message du client en message `user`. Notez le temps par ticket, et lisez deux ou trois réponses : que fait le modèle de la charte qu'il ne connaît pas ?

**Documentation utile**

- [Les chat templates](https://huggingface.co/docs/transformers/chat_templating)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`modele.generate`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationMixin.generate)
- [`tokenizer.batch_decode`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.batch_decode)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False)` : le prompt en texte, terminé par le début de la réponse de l'assistant ; `enable_thinking=False` : Qwen3 répond sans phase de réflexion
- `tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)` : un lot de prompts (le *padding* à gauche est déjà réglé)
- `modele.generate(**entrees, max_new_tokens=384, do_sample=False, temperature=None, top_p=None, top_k=None)`, sous `torch.no_grad()` : renvoie le prompt suivi de la réponse, donc gardez les tokens après `entrees.input_ids.shape[1]` ; les trois `None` retirent les réglages de tirage de Qwen3, sans effet ici, sur lesquels transformers afficherait un avertissement
- `tokenizer.batch_decode(sorties, skip_special_tokens=True)` : les textes du lot
- le temps par ticket : `(time.time() - debut) / len(test_part)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def genere_tout(conversations: list[list[dict]], taille_lot: int = 8) -> list[str]:
  textes = []
  for debut in range(0, len(conversations), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=384,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(sorties[:, ...], skip_special_tokens=True)
  return textes


def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": ...},
    {"role": "user", "content": ...},
  ]


debut = time.time()
sorties_seule = genere_tout([conversation_seule(m) for m in test_part["message"]])
print(f"{(time.time() - debut) / len(test_part):.1f} s par ticket")
print(evalue(sorties_seule))
print(sorties_seule[0])
```

### Solution

In [ ]:
def genere_tout(conversations: list[list[dict]], taille_lot: int = 8) -> list[str]:
  textes = []
  for debut in range(0, len(conversations), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=384,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": CONSIGNES},
    {"role": "user", "content": message},
  ]


debut = time.time()
sorties_seule = genere_tout([conversation_seule(m) for m in test_part["message"]])
print(f"{(time.time() - debut) / len(test_part):.1f} s par ticket")
print(evalue(sorties_seule))
print(sorties_seule[0])

## 2. Quelques exemples

Trois tickets de l'export, bien traités, servent d'exemples : `exemples`, ci-dessous, avec la réponse attendue de chacun en JSON. Écrivez `conversation_exemples(message)`, qui place avant le message du client, après `CONSIGNES`, chaque exemple en deux tours : son message en `user`, son JSON en `assistant`. Mesurez, et comparez à l'exercice 1 : qu'est-ce que les exemples ont appris au modèle, sans entraînement ?

**Documentation utile**

- [`json.dumps`](https://docs.python.org/3/library/json.html#json.dumps)

In [ ]:
EXEMPLES_IDS = ["T-13559", "T-13060", "T-12282"]
exemples = tickets.set_index("ticket_id").loc[EXEMPLES_IDS]
exemples[["message", "categorie", "priorite", "reponse"]]

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Le JSON attendu d'un exemple : `json.dumps({"categorie": ..., "priorite": ..., "reponse": ...}, ensure_ascii=False)` ; `ensure_ascii=False` garde les accents tels quels, comme le modèle doit les écrire
- `exemples.itertuples()` parcourt les lignes : `ligne.message`, `ligne.categorie`…

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def conversation_exemples(message: str) -> list[dict]:
  conversation = [{"role": "system", "content": CONSIGNES}]
  for ligne in exemples.itertuples():
    attendu = {
      "categorie": ligne.categorie,
      "priorite": ligne.priorite,
      "reponse": ligne.reponse,
    }
    conversation += [
      {"role": "user", "content": ...},
      {"role": "assistant", "content": json.dumps(..., ensure_ascii=False)},
    ]
  return conversation + [{"role": "user", "content": message}]


sorties_exemples = genere_tout([conversation_exemples(m) for m in test_part["message"]])
print(evalue(sorties_exemples))
print(sorties_exemples[0])
```

### Solution

In [ ]:
def conversation_exemples(message: str) -> list[dict]:
  conversation = [{"role": "system", "content": CONSIGNES}]
  for ligne in exemples.itertuples():
    attendu = {
      "categorie": ligne.categorie,
      "priorite": ligne.priorite,
      "reponse": ligne.reponse,
    }
    conversation += [
      {"role": "user", "content": ligne.message},
      {"role": "assistant", "content": json.dumps(attendu, ensure_ascii=False)},
    ]
  return conversation + [{"role": "user", "content": message}]


sorties_exemples = genere_tout([conversation_exemples(m) for m in test_part["message"]])
print(evalue(sorties_exemples))
print(sorties_exemples[0])

## 3. La charte dans le prompt

Le meilleur prompt qu'on puisse écrire sans entraîner : `CONSIGNES` suivies de la charte complète (`charte`), puis les trois exemples. Écrivez `conversation_charte(message)` et mesurez. C'est la barre que l'affinage devra passer, avec la seule consigne courte de l'exercice 1.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Reprenez `conversation_exemples`, et remplacez le contenu de son premier message, le message `system`, par `CONSIGNES + "\n\n" + charte`.

```python
def conversation_charte(message: str) -> list[dict]:
  conversation = conversation_exemples(message)
  conversation[0] = {"role": "system", "content": ...}
  return conversation


sorties_charte = genere_tout([conversation_charte(m) for m in test_part["message"]])
print(evalue(sorties_charte))
print(sorties_charte[0])
```

### Solution

In [ ]:
def conversation_charte(message: str) -> list[dict]:
  conversation = conversation_exemples(message)
  conversation[0] = {"role": "system", "content": CONSIGNES + "\n\n" + charte}
  return conversation


sorties_charte = genere_tout([conversation_charte(m) for m in test_part["message"]])
print(evalue(sorties_charte))
print(sorties_charte[0])

## 4. Ce que coûte chaque prompt

Le service reçoit environ 3 000 messages par mois. Pour chaque approche, comptez les tokens du prompt d'un ticket (moyenne sur `test_part`), et rassemblez dans un tableau, une ligne par approche, ces tokens, leur total par mois et les mesures des exercices 1 à 3. Le modèle affiné recevra le prompt de l'exercice 1 : que lui faudra-t-il atteindre pour valoir l'effort ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tokenizer.apply_chat_template(conversation, add_generation_prompt=True, enable_thinking=False)` renvoie, sans `tokenize=False`, les tokens du prompt : `len(...["input_ids"])`
- `pd.DataFrame(lignes, index=[...])` à partir d'une liste de dictionnaires, `evalue(...).to_dict()` pour les mesures

```python
def tokens_prompt(conversation: list[dict]) -> int:
  return len(
    tokenizer.apply_chat_template(
      conversation, add_generation_prompt=True, enable_thinking=False
    )["input_ids"]
  )


approches = {
  "consigne seule": (conversation_seule, sorties_seule),
  "exemples": (conversation_exemples, sorties_exemples),
  "charte et exemples": (conversation_charte, sorties_charte),
}
lignes = []
for nom, (fabrique, sorties) in approches.items():
  tokens = sum(tokens_prompt(fabrique(m)) for m in test_part["message"]) / len(
    test_part
  )
  lignes.append({"tokens_prompt": round(tokens), "tokens_mois": ..., **...})
pd.DataFrame(lignes, index=list(approches))
```

### Solution

In [ ]:
def tokens_prompt(conversation: list[dict]) -> int:
  return len(
    tokenizer.apply_chat_template(
      conversation, add_generation_prompt=True, enable_thinking=False
    )["input_ids"]
  )


approches = {
  "consigne seule": (conversation_seule, sorties_seule),
  "exemples": (conversation_exemples, sorties_exemples),
  "charte et exemples": (conversation_charte, sorties_charte),
}
lignes = []
for nom, (fabrique, sorties) in approches.items():
  tokens = sum(tokens_prompt(fabrique(m)) for m in test_part["message"]) / len(
    test_part
  )
  lignes.append(
    {
      "tokens_prompt": round(tokens),
      "tokens_mois": round(tokens * 3000),
      **evalue(sorties).to_dict(),
    }
  )
pd.DataFrame(lignes, index=list(approches))

## À retenir

- La consigne seule donne le format JSON, mais ni les règles de la charte ni les catégories telles que l'équipe les emploie.
- Les exemples et la charte rapprochent le modèle de ce qu'on attend, au prix d'un prompt bien plus long, payé à chaque message.
- Ces mesures sont le point de référence : le modèle affiné, avec la consigne courte, devra faire au moins aussi bien que le prompt de l'exercice 3. Et les règles vérifiables ne disent pas si la réponse est juste : l'atelier d'évaluation y ajoutera un LLM juge.